<a href="https://colab.research.google.com/github/ThanasisDoum/Nike-Sales-Analysis/blob/main/Nike_Sales_Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
import pandas as pd

#Insertion of our dataset using Google Drive
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/Nike_Project/Nike_Sales_Uncleaned.csv'
df = pd.read_csv(file_path)
df.head()

#1st cleanup step:We find the typos in the column 'Region'

# Strip leading and trailing whitespace from text columns
text_columns = ['Region', 'Gender_Category', 'Product_Line', 'Sales_Channel', 'Product_Name']
for col in text_columns:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

#By running this comamand we get the unique values of the column 'Region'
#This will be how we'll find all the typos so we can correct them
print("--- Regions Before Typos Removal ---")
print(df['Region'].unique())

# We got: 'bengaluru' 'Hyd' 'Mumbai' 'Pune' 'Delhi' 'Bangalore' 'Hyderabad' 'hyderbad' 'Kolkata'
# Now we take the wrong values, find their true counterparts and add them to this dictionary
region_map = {
    'bengaluru': 'Bangalore',
    'Hyd': 'Hyderabad',
    'hyderbad': 'Hyderabad'
}

#We replace the corrected values
df['Region'] = df['Region'].replace(region_map)

#Let's check what we did!
print("--- Regions After Typos Removal ---")
print(df['Region'].unique())

# 2nd cleanup step: We turn the 'Date' column into date format
# By using the format 'mixed' we turn every format into date format
# The errors='coerce' part turns non-valid dates into NaT (Not a Time)
df['Order_Date'] = pd.to_datetime(df['Order_Date'], format='mixed', errors='coerce')

# Calculating and storing the number of blank or invalid dates
null_dates_count = df['Order_Date'].isnull().sum()
print(f"Non valid/blank dates found: {null_dates_count}")

# Deleting rows that do not have a valid date
df = df.dropna(subset=['Order_Date'])
print("\nNew data type for the olumn Order_Date:", df['Order_Date'].dtype)

# 3rd cleanup step: We need to handle the discounts column

#if a discount is over 100%,we change it to 100%
df.loc[df['Discount_Applied'] > 1.0, 'Discount_Applied'] = 1.0

#Units sold must be a positive number
df['Units_Sold'] = df['Units_Sold'].abs()

# Fill missing MRP with the median price of the same product
df['MRP'] = df['MRP'].fillna(df.groupby('Product_Name')['MRP'].transform('median'))

# If it can't find any information about the product, it uses the average price for the entire store
df['MRP'] = df['MRP'].fillna(df['MRP'].median())

# Calculating Revenue by Simple Multiplication
df['Revenue'] = df['Units_Sold'] * df['MRP'] * (1 - df['Discount_Applied'])
df['Revenue'] = df['Revenue'].round(2)

#Removing duplicates
df = df.drop_duplicates()

#Exporting the clean csv file
clean_file_path = '/content/drive/MyDrive/Nike_Project/Nike_Sales_Cleaned.csv'
df.to_csv(clean_file_path, index=False)

Mounted at /content/drive
--- Regions Before Typos Removal ---
['bengaluru' 'Hyd' 'Mumbai' 'Pune' 'Delhi' 'Bangalore' 'Hyderabad'
 'hyderbad' 'Kolkata']
--- Regions After Typos Removal ---
['Bangalore' 'Hyderabad' 'Mumbai' 'Pune' 'Delhi' 'Kolkata']
Non valid/blank dates found: 616

New data type for the olumn Order_Date: datetime64[ns]


In [3]:
import duckdb

# Load the clean CSV as an SQL table named 'nike_sales'
csv_path = '/content/drive/MyDrive/Nike_Project/Nike_Sales_Cleaned.csv'
duckdb.sql(f"CREATE OR REPLACE TABLE nike_sales AS SELECT * FROM read_csv_auto('{csv_path}')")

# 3. Επιβεβαίωση: Εμφάνιση των πρώτων 5 εγγραφών
duckdb.sql("SELECT * FROM nike_sales LIMIT 5").df()

# Pillar 1: High-Level Business KPIs
#Here we calculate
query_kpis = """
SELECT
    ROUND(SUM(Revenue), 2) AS Total_Revenue,
    SUM(Units_Sold) AS Total_Units_Sold,
    COUNT(*) AS Total_Orders,
    ROUND(AVG(Revenue), 2) AS Average_Order_Value
FROM nike_sales;
"""

# Execute the SQL query and display the result
df_kpis = duckdb.sql(query_kpis).df()
print("--- OVERALL BUSINESS KPIs ---")
df_kpis

--- OVERALL BUSINESS KPIs ---


,Total_Revenue,Total_Units_Sold,Total_Orders,Average_Order_Value
0,1448723.99,1765.0,1884,4298.88


In [5]:
# Pillar 2: Regional Sales Performance
query_region = """
SELECT
    Region,
    ROUND(SUM(Revenue), 2) AS Total_Revenue,
    SUM(Units_Sold) AS Total_Units_Sold,
    COUNT(*) AS Total_Orders,
    ROUND(AVG(Revenue), 2) AS Avg_Order_Value
FROM nike_sales
GROUP BY Region
ORDER BY Total_Revenue DESC;
"""

# Execute the SQL query and display the result
df_region = duckdb.sql(query_region).df()
print("--- REGIONAL PERFORMANCE ---")
df_region

--- REGIONAL PERFORMANCE ---


,Region,Total_Revenue,Total_Units_Sold,Total_Orders,Avg_Order_Value
0,Hyderabad,284687.99,265.0,309,5272.00
1,Kolkata,282795.52,276.0,317,5545.01
2,Bangalore,249232.53,295.0,324,3719.89
3,Delhi,233258.97,285.0,322,3823.92
4,Pune,202392.15,292.0,290,4047.84
5,Mumbai,196356.83,352.0,322,3636.24


In [6]:
# Pillar 3: Top 5 Best-Selling Products by Revenue
query_top_products = """
SELECT
    Product_Name,
    ROUND(SUM(Revenue), 2) AS Total_Revenue,
    SUM(Units_Sold) AS Total_Units_Sold,
    ROUND(AVG(Discount_Applied) * 100, 2) AS Avg_Discount_Percent
FROM nike_sales
GROUP BY Product_Name
ORDER BY Total_Revenue DESC
LIMIT 5;
"""

# Pillar 3: Bottom 5 Lowest-Performing Products by Revenue
query_bottom_products = """
SELECT
    Product_Name,
    ROUND(SUM(Revenue), 2) AS Total_Revenue,
    SUM(Units_Sold) AS Total_Units_Sold,
    ROUND(AVG(Discount_Applied) * 100, 2) AS Avg_Discount_Percent
FROM nike_sales
GROUP BY Product_Name
ORDER BY Total_Revenue ASC
LIMIT 5;
"""


#Execute the SQL queries and display the result
print("--- TOP 5 PRODUCTS BY REVENUE ---")
df_top = duckdb.sql(query_top_products).df()
display(df_top)

print("\n--- BOTTOM 5 PRODUCTS BY REVENUE ---")
df_bottom = duckdb.sql(query_bottom_products).df()
display(df_bottom)

--- TOP 5 PRODUCTS BY REVENUE ---


,Product_Name,Total_Revenue,Total_Units_Sold,Avg_Discount_Percent
0,LeBron 20,124875.01,96.0,57.86
1,Zoom Freak,120655.49,104.0,59.00
2,Metcon 7,104023.48,84.0,49.12
3,Flex Trainer,95408.80,99.0,56.46
4,Phantom GT,94787.57,102.0,57.21



--- BOTTOM 5 PRODUCTS BY REVENUE ---


,Product_Name,Total_Revenue,Total_Units_Sold,Avg_Discount_Percent
0,Air Zoom,32816.00,62.0,54.24
1,Pegasus Turbo,37999.39,45.0,62.17
2,Waffle One,38623.44,91.0,69.44
3,Dunk Low,44840.61,123.0,64.03
4,Kyrie Flytrap,44865.62,58.0,61.04


In [7]:
# Pillar 4: Discount Analysis by Product Line & Sales Channel
query_discounts = """
SELECT
    Product_Line,
    Sales_Channel,
    COUNT(*) AS Total_Orders,
    SUM(Units_Sold) AS Total_Units_Sold,
    ROUND(SUM(Revenue), 2) AS Total_Revenue,
    ROUND(AVG(Discount_Applied) * 100, 2) AS Avg_Discount_Percent
FROM nike_sales
GROUP BY Product_Line, Sales_Channel
ORDER BY Avg_Discount_Percent DESC;
"""

# Execute the SQL query and display the result
df_discounts = duckdb.sql(query_discounts).df()
print("---  DISCOUNT ANALYSIS BY CATEGORY & CHANNEL ---")
df_discounts

---  DISCOUNT ANALYSIS BY CATEGORY & CHANNEL ---


,Product_Line,Sales_Channel,Total_Orders,Total_Units_Sold,Total_Revenue,Avg_Discount_Percent
0,Lifestyle,Retail,203,206.0,107766.74,69.76
1,Running,Online,175,142.0,99862.53,62.66
2,Soccer,Retail,176,177.0,125022.73,61.68
3,Lifestyle,Online,185,192.0,89798.54,61.19
4,Basketball,Online,198,168.0,179031.79,61.03
5,Running,Retail,166,126.0,91004.48,60.47
6,Soccer,Online,186,191.0,192102.09,57.68
7,Training,Online,202,209.0,195619.52,57.36
8,Basketball,Retail,198,176.0,198359.88,56.44
9,Training,Retail,195,178.0,170155.69,55.29


In [8]:
# Pillar 5: Sales Channel Comparison (Online vs In-Store vs Outlet)
query_channels = """
SELECT
    Sales_Channel,
    COUNT(*) AS Total_Orders,
    SUM(Units_Sold) AS Total_Units_Sold,
    ROUND(SUM(Revenue), 2) AS Total_Revenue,
    ROUND(AVG(Revenue), 2) AS Average_Order_Value,
    ROUND(AVG(Discount_Applied) * 100, 2) AS Avg_Discount_Percent,
    ROUND(SUM(Revenue) * 100.0 / (SELECT SUM(Revenue) FROM nike_sales), 2) AS Revenue_Share_Percent
FROM nike_sales
GROUP BY Sales_Channel
ORDER BY Total_Revenue DESC;
"""

# Execute the SQL query and display the result
df_channels = duckdb.sql(query_channels).df()
print("--- 🛒 SALES CHANNEL PERFORMANCE COMPARISON ---")
df_channels

--- 🛒 SALES CHANNEL PERFORMANCE COMPARISON ---


,Sales_Channel,Total_Orders,Total_Units_Sold,Total_Revenue,Average_Order_Value,Avg_Discount_Percent,Revenue_Share_Percent
0,Online,946,902.0,756414.47,4273.53,59.85,52.21
1,Retail,938,863.0,692309.52,4326.93,60.58,47.79
